In [112]:
import  torch
from  d2l    import  torch as d2l

# 多输入通道卷积（multi-in convolution）
# 该函数实现对多输入通道的卷积操作，其数学含义为：
#     Y = X1 * K1 + X2 * K2 + ... + Xc * Kc
# 输入：
#   X: 输入张量，形状为 (in_channels, H, W)
#   K: 卷积核张量，形状为 (in_channels, kH, kW)
# 实现逻辑：
#   1. zip(X, K) 将每个输入通道与对应卷积核通道一一配对
#   2. d2l.corr2d(x, k) 对每对 (x, k) 执行 2D 互相关运算
#   3. sum(...) 将所有通道卷积结果逐元素相加，得到最终单通道输出
# 输出：
#   返回一个 2D 张量（大小为 (H - kH + 1, W - kW + 1)）
def corr2d_multi_in(X,K):

    return  sum(d2l.corr2d(x,k) for x,k in zip(X,K))



In [113]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

In [114]:
# 多输入多输出通道卷积
# X: 输入张量，形状为 (in_channels, H, W)
# K: 卷积核张量，形状为 (out_channels, in_channels, kH, kW)
def corr2d_multi_in_out(X, K):
    # 对 K 的第 0 维进行遍历，每个 k 是一个卷积核（含多个输入通道）
    # 对每个卷积核执行多输入通道卷积 corr2d_multi_in(X, k)
    # 最后将所有输出通道堆叠到一起，形成多输出通道的结果
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

In [115]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

In [116]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

In [117]:
# 这是一个用于演示 1x1 卷积的函数
def corr2d_multi_in_out_1x1(X, K):
    # 1. 获取输入张量 X 的形状信息
    # c_i 是输入通道数, h 是高度, w 是宽度
    c_i, h, w = X.shape

    # 2. 获取卷积核 K 的形状信息
    # c_o 是输出通道数
    c_o = K.shape[0]

    # 3. 将输入张量 X 变形
    # 从 (c_i, h, w) 变为 (c_i, h * w)
    X = X.reshape((c_i, h * w))

    # 4. 将卷积核 K 变形
    # K 的原始形状是 (c_o, c_i, 1, 1)，变为 (c_o, c_i)
    K = K.reshape((c_o, c_i))

    # 5. 执行矩阵乘法
    # 这等效于一个全连接层的计算
    Y = torch.matmul(K, X)

    # 6. 将输出张量 Y 变形回目标形状
    # 从 (c_o, h * w) 变为 (c_o, h, w)
    return Y.reshape((c_o, h, w))

In [118]:
# 创建一个形状为 (3, 3, 3) 的随机输入张量 X
# 代表 3 个输入通道，高度和宽度都为 3
X = torch.normal(0, 1, (3, 3, 3))

# 创建一个形状为 (2, 3, 1, 1) 的随机卷积核张量 K
# 代表 2 个输出通道，3 个输入通道，卷积核高宽为 1x1
K = torch.normal(0, 1, (2, 3, 1, 1))

# 使用 1x1 卷积函数计算输出 Y1
Y1 = corr2d_multi_in_out_1x1(X, K)

# 使用通用的多输入多输出卷积函数计算输出 Y2
Y2 = corr2d_multi_in_out(X, K)

# 断言：检查两种方法计算的结果是否在误差允许范围内（atol=1e-6）非常接近
assert torch.allclose(Y1, Y2, atol=1e-6)